### Step 1: Load Libraries

In [2]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display
from pprint import pprint
import json

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if OPENAI_API_KEY is None:
    raise Exception("API Key is missing!")
else:
    print("Key is: " + OPENAI_API_KEY[:8])

Key is: sk-proj-


### Step 2: Setup PushOver

In [3]:
load_dotenv()

pushover_user  = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url   = "https://api.pushover.net/1/messages.json"


In [4]:
#Test Pushover
import requests

def send_notification(message: str):
    pay_load = {"user":pushover_user,
                "token": pushover_token,
                "message": message}
    requests.post(pushover_url, data = pay_load)

send_notification("Hello to Si Lam")
    

### Step 3: Describe Pushover in LLM

In [5]:
send_notification_function = {

    "name" : "send_notification",
    "description": "Send a push notification to user phone via pushover. Tell the user important task, event",
    "parameters": {
        "type": "object",
        "properties":{
            "message": {
                "type": "string",
                "description":"The notification message to send to user device"
            }
        },
        "required": ["message"]
    }
}

### Step 4: Add Pushover to the list of tools for LLM

In [6]:
tools = [
    {"type": "function","function": send_notification_function}
]


### Create a new function and add it to the list of tools

In [7]:
import random

# simulate rolling a dice
def dice_roll():
    result = random.randint(1,6)
    return result 

In [8]:
roll_dice_function = {
    "name" : "dice_roll",
    "description": "Return a number by rolling a dice",
    "parameters": {
        "type": "object",
        "properties":{
            
        },
        "required": []
    }
}

tools.append( {"type": "function","function": roll_dice_function})

In [9]:
from email import message


def handle_tool_call(tool_calls):
    tools_results = []
    # return what to add to our context (about tool call results, a dictionary)
    for tool_call in tool_calls:
        function_name = tool_call.function.name
        print(f"Calling function {function_name}")
        args = json.loads(tool_call.function.arguments)

        #Route to function name
        if function_name == "send_notification":

            send_notification(args['message'])
            print(f"Send notification: {args['message']}")
            content = f"Notitication sent: {args['message']}"
        elif function_name == "dice_roll":

            #function 2(args['message'])
            #print(f"Send notification: {args['message']}")
            content = f"Rolled: {dice_roll()}"
        else:
            content = f"Unknown function {function_name}"

        tool_call_result  = {
            "role":"tool",
            "content":content,
            "tool_call_id":tool_call.id
            
        }

        tools_results.append(tool_call_result)

    return tools_results

### Step 5: calling the tool from LLM

In [10]:
client = OpenAI()
messages=[
        {"role":"user", "content":"Please do two things \
         1)- I want to roll 2 dice and \
         2) - Send notification with the highest dice result"}
    ]
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages,
    tools= tools,
    tool_choice="auto"
)

#check if the model wants to call a tool

message = response.choices[0].message

tool_loop_active = True
finish_response = ''
while message.tool_calls:
    #handle tool call
    tool_result = handle_tool_call(message.tool_calls) #whole list of tools
    pprint("tool_result:" + str(tool_result))

    #add message to context
    messages.append(message)

    #add info about tool call response to context, i.e. mesages
    messages.extend(tool_result)
    pprint("message append tool_result:" + str(message))

    #invokde LLM one more time to get its updated response
    response = client.chat.completions.create(
        model='gpt-4.1-mini',
        messages=messages,
        tools=tools #will be in the future
    )
    message = response.choices[0].message
    

    # print(message.content) from new LLM response
   
pprint(message)

Calling function dice_roll
Calling function dice_roll
("tool_result:[{'role': 'tool', 'content': 'Rolled: 4', 'tool_call_id': "
 "'call_8v3bHk1gyT95NIFowQwEI1qx'}, {'role': 'tool', 'content': 'Rolled: 5', "
 "'tool_call_id': 'call_JMYMwMxzbg8XV807XNANulGD'}]")
('message append tool_result:ChatCompletionMessage(content=None, refusal=None, '
 "role='assistant', annotations=[], audio=None, function_call=None, "
 "tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_8v3bHk1gyT95NIFowQwEI1qx', "
 "function=Function(arguments='{}', name='dice_roll'), type='function'), "
 "ChatCompletionMessageFunctionToolCall(id='call_JMYMwMxzbg8XV807XNANulGD', "
 "function=Function(arguments='{}', name='dice_roll'), type='function')])")
Calling function send_notification
Send notification: The highest dice result rolled is 5.
("tool_result:[{'role': 'tool', 'content': 'Notitication sent: The highest "
 "dice result rolled is 5.', 'tool_call_id': 'call_S4fhn42GLi8E2YXmnreIhero'}]")
('message append too